# 第14章　前処理パイプラインの完全実装 ― 生画像から学習可能なテンソルまで**『本格実装 医療診断支援AI（実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-impl

## 14.1　2次元CT ― DICOMからテンソルへ

In [ ]:
import pydicom, numpy as np, torchdef preprocess_ct_2d(dcm_path, window=(40, 400), size=512):    ds = pydicom.dcmread(dcm_path)    # ① 生画素をHU値に変換    hu = ds.pixel_array * ds.RescaleSlope + ds.RescaleIntercept    # ② ウィンドウでクリップし、0〜1へ正規化（基礎編のレシピ集の章）    lo, hi = window[0] - window[1]/2, window[0] + window[1]/2    img = np.clip(hu, lo, hi)    img = (img - lo) / (hi - lo)    # ③ サイズをそろえ、テンソル化（チャネル次元を付ける）    import cv2    img = cv2.resize(img.astype(np.float32), (size, size))    return torch.from_numpy(img).unsqueeze(0)        # (1, H, W)

## 14.2　3次元 ― MONAIで一気通貫

In [ ]:
from monai import transforms as Tdef build_pipeline(train=True):    steps = [        T.LoadImaged(keys=["image", "label"]),                       # NIfTI読み込み        T.EnsureChannelFirstd(keys=["image", "label"]),        T.Orientationd(keys=["image", "label"], axcodes="RAS"),      # ① 向きを統一        T.Spacingd(keys=["image", "label"], pixdim=(1.5,)*3,                   mode=("bilinear", "nearest")),                    # ② 解像度統一        T.ScaleIntensityRanged(keys="image", a_min=-100, a_max=240,                               b_min=0.0, b_max=1.0, clip=True),     # ③ HUクリップ正規化        T.CropForegroundd(keys=["image", "label"], source_key="image"),  # ④ 余白除去    ]    if train:                                                        # ⑤ 学習時のみ拡張        steps += [            T.SpatialPadd(keys=["image", "label"], spatial_size=(96,)*3, mode="constant"),            T.RandCropByPosNegLabeld(keys=["image","label"], label_key="label",                spatial_size=(96,)*3, pos=3, neg=1, num_samples=4),  # 前景中心を優先（前景に臓器も含まれる点に注意）            # RAS前提なので spatial_axis=2 は頭尾方向の反転。実際には撮影されえない向きだが、            # nnU-Net が既定で用いる鏡像反転にならった設定（第16章）。採否は検証データで確かめる。            T.RandFlipd(keys=["image","label"], prob=0.5, spatial_axis=2),            T.RandGaussianNoised(keys="image", prob=0.2),        ]    return T.Compose(steps)train_tf, val_tf = build_pipeline(train=True), build_pipeline(train=False)

## 14.3　検証を、必ず挟む

In [ ]:
data = val_tf({"image": "case_001.nii.gz", "label": "case_001_seg.nii.gz"})  # 検証は拡張なしのval_tfで（train_tfはパッチを複数返す）img, lbl = data["image"][0], data["label"][0]assert img.shape == lbl.shape, "画像とラベルの形が不一致"print(f"値域: {img.min():.2f}〜{img.max():.2f}（0〜1のはず）")# さらに、代表断面にマスクを重ねて目視（基礎編の可視化の章の show_3d）